In [23]:
# Standard libraries
import os
import sys
import json
from pathlib import Path

# Data analysis
import pandas as pd

# Notebook display|
from IPython.display import display

# PySpark
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.sql import types as T

In [24]:
# Project directories
PROJECT_ROOT = Path(r"D:\projects\AI_Monitoring_Project")

ADVANCED_DIR = PROJECT_ROOT / "advanced"
ETL_DIR = ADVANCED_DIR / "ETL"

# Warehouse directories
WAREHOUSE_DIR = ETL_DIR / "warehouse"

BRONZE_WAREHOUSE_DIR = WAREHOUSE_DIR / "bronze"
SILVER_WAREHOUSE_DIR = WAREHOUSE_DIR / "silver"
GOLD_WAREHOUSE_DIR = WAREHOUSE_DIR / "gold"

# Spark directories
SPARK_TEMP_DIR = ETL_DIR / ".spark-temp"
METASTORE_DB_DIR = ETL_DIR / "metastore_db"

# Create required directories
for directory in [
    SPARK_TEMP_DIR,
    SILVER_WAREHOUSE_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("ETL directory:", ETL_DIR)
print("Warehouse:", WAREHOUSE_DIR)
print("Bronze:", BRONZE_WAREHOUSE_DIR)
print("Silver:", SILVER_WAREHOUSE_DIR)
print("Gold:", GOLD_WAREHOUSE_DIR)

Project root: D:\projects\AI_Monitoring_Project
ETL directory: D:\projects\AI_Monitoring_Project\advanced\ETL
Warehouse: D:\projects\AI_Monitoring_Project\advanced\ETL\warehouse
Bronze: D:\projects\AI_Monitoring_Project\advanced\ETL\warehouse\bronze
Silver: D:\projects\AI_Monitoring_Project\advanced\ETL\warehouse\silver
Gold: D:\projects\AI_Monitoring_Project\advanced\ETL\warehouse\gold


In [25]:
# Java configuration
JAVA_HOME = Path(
    r"C:\Program Files\Eclipse Adoptium\jdk-17.0.20.101-hotspot"
)

# Hadoop configuration
HADOOP_HOME = Path(
    r"C:\hadoop\hadoop-3.5.0"
)

# Verify required files
required_files = [
    JAVA_HOME / "bin" / "java.exe",
    HADOOP_HOME / "bin" / "winutils.exe",
    HADOOP_HOME / "bin" / "hadoop.dll",
    HADOOP_HOME / "bin" / "hdfs.dll"
]

for file_path in required_files:
    if not file_path.exists():
        raise FileNotFoundError(
            f"Required file not found: {file_path}"
        )

# Environment variables
os.environ["JAVA_HOME"] = str(JAVA_HOME)
os.environ["HADOOP_HOME"] = str(HADOOP_HOME)

# Update PATH
os.environ["PATH"] = os.pathsep.join([
    str(JAVA_HOME / "bin"),
    str(HADOOP_HOME / "bin"),
    os.environ.get("PATH", "")
])

print("Java and Hadoop configured successfully.")

Java and Hadoop configured successfully.


In [26]:
# Java and Hadoop configuration

java_home = Path(
    r"C:\Program Files\Eclipse Adoptium\jdk-17.0.20.101-hotspot"
)

hadoop_home = Path(
    r"C:\hadoop\hadoop-3.5.0"
)

# Verify required files
required_files = [
    java_home / "bin" / "java.exe",
    hadoop_home / "bin" / "winutils.exe",
    hadoop_home / "bin" / "hadoop.dll",
    hadoop_home / "bin" / "hdfs.dll"
]

for file in required_files:
    if not file.exists():
        raise FileNotFoundError(f"Missing file: {file}")

# Configure environment variables
os.environ["JAVA_HOME"] = str(java_home)
os.environ["HADOOP_HOME"] = str(hadoop_home)

# Update PATH
os.environ["PATH"] = os.pathsep.join([
    str(java_home / "bin"),
    str(hadoop_home / "bin"),
    os.environ.get("PATH", "")
])

print("Java configured:", java_home)
print("Hadoop configured:", hadoop_home)
print("Hadoop utilities verified.")

Java configured: C:\Program Files\Eclipse Adoptium\jdk-17.0.20.101-hotspot
Hadoop configured: C:\hadoop\hadoop-3.5.0
Hadoop utilities verified.


In [27]:
# Derby metastore connection
metastore_url = (
    f"jdbc:derby:;databaseName="
    f"{METASTORE_DB_DIR.as_posix()};create=true"
)

# Initialize SparkSession
spark = (
    SparkSession.builder
    .appName("AI_Monitoring_Silver")
    .master("local[2]")
    .config(
        "spark.sql.warehouse.dir",
        WAREHOUSE_DIR.as_uri()
    )
    .config(
        "spark.local.dir",
        str(SPARK_TEMP_DIR)
    )
    .config(
        "spark.sql.catalogImplementation",
        "hive"
    )
    .config(
        "spark.hadoop.javax.jdo.option.ConnectionURL",
        metastore_url
    )
    .enableHiveSupport()
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print("Spark version:", spark.version)

print(
    "Hadoop version:",
    spark.sparkContext._jvm
    .org.apache.hadoop.util.VersionInfo.getVersion()
)

Spark version: 4.2.0
Hadoop version: 3.5.0


In [28]:
print("Silver Metastore URL:", metastore_url)
print("Silver Warehouse:", WAREHOUSE_DIR)

Silver Metastore URL: jdbc:derby:;databaseName=D:/projects/AI_Monitoring_Project/advanced/ETL/metastore_db;create=true
Silver Warehouse: D:\projects\AI_Monitoring_Project\advanced\ETL\warehouse


In [29]:
# Spark and Hive diagnostics

print("=" * 60)
print("SPARK DIAGNOSTICS")
print("=" * 60)

print("Spark version:", spark.version)

print(
    "Catalog implementation:",
    spark.conf.get("spark.sql.catalogImplementation")
)

print(
    "Warehouse directory:",
    spark.conf.get("spark.sql.warehouse.dir")
)

print("\n" + "=" * 60)
print("METASTORE DIAGNOSTICS")
print("=" * 60)

print("Expected metastore URL:")
print(metastore_url)

print("\nConfigured metastore URL:")

configured_url = (
    spark.sparkContext._jsc
    .hadoopConfiguration()
    .get("javax.jdo.option.ConnectionURL")
)

print(configured_url)

print("\nMetastore directory:")
print(METASTORE_DB_DIR.resolve())

print("Directory exists:", METASTORE_DB_DIR.exists())

print("\nMetastore directory contents:")

if METASTORE_DB_DIR.exists():
    for item in list(METASTORE_DB_DIR.iterdir())[:15]:
        print(item.name)

SPARK DIAGNOSTICS
Spark version: 4.2.0
Catalog implementation: hive
Warehouse directory: file:/D:/projects/AI_Monitoring_Project/advanced/ETL/warehouse

METASTORE DIAGNOSTICS
Expected metastore URL:
jdbc:derby:;databaseName=D:/projects/AI_Monitoring_Project/advanced/ETL/metastore_db;create=true

Configured metastore URL:
jdbc:derby:;databaseName=D:/projects/AI_Monitoring_Project/advanced/ETL/metastore_db;create=true

Metastore directory:
D:\projects\AI_Monitoring_Project\advanced\ETL\metastore_db
Directory exists: True

Metastore directory contents:
db.lck
log
README_DO_NOT_TOUCH_FILES.txt
seg0
service.properties
tmp


In [8]:
display(
    spark.sql("SHOW DATABASES").toPandas()
)

,namespace
0,bronze
1,default
2,gold
3,silver


In [9]:
# Create Silver database
spark.sql(f"""
    CREATE DATABASE IF NOT EXISTS silver
    LOCATION '{SILVER_WAREHOUSE_DIR.as_uri()}'
""")

# Create Gold database
spark.sql(f"""
    CREATE DATABASE IF NOT EXISTS gold
    LOCATION '{GOLD_WAREHOUSE_DIR.as_uri()}'
""")

print("Silver and Gold databases are ready.")

Silver and Gold databases are ready.


In [10]:
display(
    spark.sql("SHOW DATABASES").toPandas()
)

,namespace
0,bronze
1,default
2,gold
3,silver


In [11]:
display(
    spark.sql("SHOW TABLES IN bronze").toPandas()
)

,namespace,tableName,isTemporary
0,bronze,cameras,False
1,bronze,employees,False
2,bronze,person_detections_worker1,False
3,bronze,person_detections_worker2,False
4,bronze,plate_detections,False
5,bronze,restricted_access,False


In [12]:
# Person detection datasets
person_worker1_df = spark.table(
    "bronze.person_detections_worker1"
)

person_worker2_df = spark.table(
    "bronze.person_detections_worker2"
)

# Plate detection dataset
plate_detections_df = spark.table(
    "bronze.plate_detections"
)

# Reference datasets
employees_df = spark.table(
    "bronze.employees"
)

cameras_df = spark.table(
    "bronze.cameras"
)

restricted_access_df = spark.table(
    "bronze.restricted_access"
)

print("All six Bronze DataFrames loaded successfully.")

All six Bronze DataFrames loaded successfully.


In [13]:
bronze_dataframes = {
    "person_worker1": person_worker1_df,
    "person_worker2": person_worker2_df,
    "plate_detections": plate_detections_df,
    "employees": employees_df,
    "cameras": cameras_df,
    "restricted_access": restricted_access_df
}

In [15]:
# Person detection JSON schemas

person_ids_schema = T.ArrayType(
    T.StringType()
)

authorization_schema = T.ArrayType(
    T.BooleanType()
)

In [16]:
person_worker1_silver_df = (
    person_worker1_df

    # Timestamp conversion
    .withColumn(
        "Time_stamp",
        F.to_timestamp("Time_stamp")
    )

    # Parse detected employee IDs
    .withColumn(
        "detected_persons_id",
        F.from_json(
            F.col("detected_persons_id"),
            person_ids_schema
        )
    )

    # Parse authorization flags
    .withColumn(
        "authorized",
        F.from_json(
            F.col("authorized"),
            authorization_schema
        )
    )
)

In [17]:
person_worker1_silver_df.printSchema()

root
 |-- Time_stamp: timestamp (nullable = true)
 |-- Zone: string (nullable = true)
 |-- detected_persons_id: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- authorized: array (nullable = true)
 |    |-- element: boolean (containsNull = true)
 |-- camera_id: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- detector_type: string (nullable = true)



In [18]:
display(person_worker1_silver_df.toPandas())

,Time_stamp,Zone,detected_persons_id,authorized,camera_id,event_id,detector_type
0,2026-10-28 12:09:58.634651,A,"[EMP0029, EMP0072]","[True, True]",001,camera.person.frames.dev:0:2637,synthetic
1,2026-10-28 12:09:58.634651,A,[EMP0088],[True],002,camera.person.frames.dev:0:2638,synthetic
2,2026-10-28 12:09:58.634651,B,[],[],003,camera.person.frames.dev:0:2639,synthetic
3,2026-10-28 12:09:59.691376,A,"[EMP0029, EMP0072]","[True, True]",001,camera.person.frames.dev:0:2640,synthetic
4,2026-10-28 12:09:59.691376,A,[EMP0088],[True],002,camera.person.frames.dev:0:2641,synthetic
...,...,...,...,...,...,...,...
16854,2026-10-28 15:51:41.090889,B,"[EMP0020, EMP0029, EMP0087]","[True, True, True]",003,camera.person.frames.dev:0:19460,synthetic
16855,2026-10-28 15:51:41.090889,B,"[EMP0017, EMP0021, EMP0072]","[True, True, True]",004,camera.person.frames.dev:1:25908,synthetic
16856,2026-10-28 15:51:41.094153,C,[],[],005,camera.person.frames.dev:1:25909,synthetic
16857,2026-10-28 15:51:41.094657,D,[],[],006,camera.person.frames.dev:1:25910,synthetic


In [19]:
person_worker2_silver_df = (
    person_worker2_df

    # Timestamp conversion
    .withColumn(
        "Time_stamp",
        F.to_timestamp("Time_stamp")
    )

    # Parse detected employee IDs
    .withColumn(
        "detected_persons_id",
        F.from_json(
            F.col("detected_persons_id"),
            person_ids_schema
        )
    )

    # Parse authorization flags
    .withColumn(
        "authorized",
        F.from_json(
            F.col("authorized"),
            authorization_schema
        )
    )
)

In [20]:
display(person_worker2_silver_df.toPandas())

,Time_stamp,Zone,detected_persons_id,authorized,camera_id,event_id,detector_type
0,2026-10-28 12:09:58.634651,B,"[EMP0031, EMP0070, EMP0087]","[True, True, True]",004,camera.person.frames.dev:1:3480,synthetic
1,2026-10-28 12:09:58.634651,C,[],[],005,camera.person.frames.dev:1:3481,synthetic
2,2026-10-28 12:09:58.634651,D,[],[],006,camera.person.frames.dev:1:3482,synthetic
3,2026-10-28 12:09:58.634651,D,[],[],007,camera.person.frames.dev:1:3483,synthetic
4,2026-10-28 12:09:59.693337,B,"[EMP0031, EMP0070, EMP0087]","[True, True, True]",004,camera.person.frames.dev:1:3484,synthetic
...,...,...,...,...,...,...,...
22392,2026-10-28 15:51:34.021367,D,[],[],007,camera.person.frames.dev:1:25883,synthetic
22393,2026-10-28 15:51:35.029525,B,"[EMP0017, EMP0021, EMP0072]","[True, True, True]",004,camera.person.frames.dev:1:25884,synthetic
22394,2026-10-28 15:51:35.030551,C,[],[],005,camera.person.frames.dev:1:25885,synthetic
22395,2026-10-28 15:51:35.031198,D,[],[],006,camera.person.frames.dev:1:25886,synthetic


In [21]:
person_detections_silver_df = (
    person_worker1_silver_df
    .unionByName(person_worker2_silver_df)
)

In [22]:
display(person_detections_silver_df.toPandas())

,Time_stamp,Zone,detected_persons_id,authorized,camera_id,event_id,detector_type
0,2026-10-28 12:09:58.634651,A,"[EMP0029, EMP0072]","[True, True]",001,camera.person.frames.dev:0:2637,synthetic
1,2026-10-28 12:09:58.634651,A,[EMP0088],[True],002,camera.person.frames.dev:0:2638,synthetic
2,2026-10-28 12:09:58.634651,B,[],[],003,camera.person.frames.dev:0:2639,synthetic
3,2026-10-28 12:09:59.691376,A,"[EMP0029, EMP0072]","[True, True]",001,camera.person.frames.dev:0:2640,synthetic
4,2026-10-28 12:09:59.691376,A,[EMP0088],[True],002,camera.person.frames.dev:0:2641,synthetic
...,...,...,...,...,...,...,...
39251,2026-10-28 15:51:34.021367,D,[],[],007,camera.person.frames.dev:1:25883,synthetic
39252,2026-10-28 15:51:35.029525,B,"[EMP0017, EMP0021, EMP0072]","[True, True, True]",004,camera.person.frames.dev:1:25884,synthetic
39253,2026-10-28 15:51:35.030551,C,[],[],005,camera.person.frames.dev:1:25885,synthetic
39254,2026-10-28 15:51:35.031198,D,[],[],006,camera.person.frames.dev:1:25886,synthetic


In [30]:
# Write frame-level person detections

(
    person_detections_silver_df
    .write
    .format("parquet")
    .mode("overwrite")
    .saveAsTable("silver.person_detections")
)

print("silver.person_detections created successfully.")

silver.person_detections created successfully.


In [31]:
# Pair employee IDs with authorization statuses

person_detection_details_df = (
    person_detections_silver_df

    .withColumn(
        "_person_details",
        F.explode(
            F.arrays_zip(
                F.col("detected_persons_id"),
                F.col("authorized")
            )
        )
    )

    .select(
        "Time_stamp",
        "Zone",
        "camera_id",
        "event_id",
        "detector_type",

        F.col(
            "_person_details.detected_persons_id"
        ).alias("employee_id"),

        F.col(
            "_person_details.authorized"
        ).alias("authorized")
    )
)

In [32]:
display(person_detection_details_df.toPandas())

,Time_stamp,Zone,camera_id,event_id,detector_type,employee_id,authorized
0,2026-10-28 12:09:58.634651,A,001,camera.person.frames.dev:0:2637,synthetic,EMP0029,True
1,2026-10-28 12:09:58.634651,A,001,camera.person.frames.dev:0:2637,synthetic,EMP0072,True
2,2026-10-28 12:09:58.634651,A,002,camera.person.frames.dev:0:2638,synthetic,EMP0088,True
3,2026-10-28 12:09:59.691376,A,001,camera.person.frames.dev:0:2640,synthetic,EMP0029,True
4,2026-10-28 12:09:59.691376,A,001,camera.person.frames.dev:0:2640,synthetic,EMP0072,True
...,...,...,...,...,...,...,...
36468,2026-10-28 15:51:34.016769,B,004,camera.person.frames.dev:1:25880,synthetic,EMP0021,True
36469,2026-10-28 15:51:34.016769,B,004,camera.person.frames.dev:1:25880,synthetic,EMP0072,True
36470,2026-10-28 15:51:35.029525,B,004,camera.person.frames.dev:1:25884,synthetic,EMP0017,True
36471,2026-10-28 15:51:35.029525,B,004,camera.person.frames.dev:1:25884,synthetic,EMP0021,True


In [33]:
person_detection_details_df.write.format("parquet").mode("overwrite").saveAsTable("silver.person_detection_details")
print("silver.person_detection_details created successfully.")

silver.person_detection_details created successfully.


In [34]:
display(plate_detections_df.toPandas())

,Time_stamp,Zone,detected_plates,camera_id,event_id,detector_type
0,2026-10-28T08:30:27.299987+00:00,E,[],008,camera.plate.frames.dev:0:1821,synthetic
1,2026-10-28T08:30:27.299987+00:00,E,[],009,camera.plate.frames.dev:0:1822,synthetic
2,2026-10-28T08:30:27.299987+00:00,E,[],010,camera.plate.frames.dev:0:1823,synthetic
3,2026-10-28T08:30:28.351386+00:00,E,[],008,camera.plate.frames.dev:0:1824,synthetic
4,2026-10-28T08:30:28.354365+00:00,E,[],009,camera.plate.frames.dev:0:1825,synthetic
...,...,...,...,...,...,...
17608,2026-10-28T12:21:40.086463+00:00,E,[],009,camera.plate.frames.dev:0:19429,synthetic
17609,2026-10-28T12:21:40.087033+00:00,E,[],010,camera.plate.frames.dev:0:19430,synthetic
17610,2026-10-28T12:21:41.095783+00:00,E,[],008,camera.plate.frames.dev:0:19431,synthetic
17611,2026-10-28T12:21:41.096319+00:00,E,[],009,camera.plate.frames.dev:0:19432,synthetic


In [35]:
plate_detections_df.printSchema()

root
 |-- Time_stamp: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- detected_plates: string (nullable = true)
 |-- camera_id: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- detector_type: string (nullable = true)



In [36]:
plate_detection_silver_df = (
    plate_detections_df

    .withColumn(
        "Time_stamp",
        F.to_timestamp("Time_stamp")
    )
)
display(plate_detection_silver_df.toPandas())



,Time_stamp,Zone,detected_plates,camera_id,event_id,detector_type
0,2026-10-28 12:00:27.299987,E,[],008,camera.plate.frames.dev:0:1821,synthetic
1,2026-10-28 12:00:27.299987,E,[],009,camera.plate.frames.dev:0:1822,synthetic
2,2026-10-28 12:00:27.299987,E,[],010,camera.plate.frames.dev:0:1823,synthetic
3,2026-10-28 12:00:28.351386,E,[],008,camera.plate.frames.dev:0:1824,synthetic
4,2026-10-28 12:00:28.354365,E,[],009,camera.plate.frames.dev:0:1825,synthetic
...,...,...,...,...,...,...
17608,2026-10-28 15:51:40.086463,E,[],009,camera.plate.frames.dev:0:19429,synthetic
17609,2026-10-28 15:51:40.087033,E,[],010,camera.plate.frames.dev:0:19430,synthetic
17610,2026-10-28 15:51:41.095783,E,[],008,camera.plate.frames.dev:0:19431,synthetic
17611,2026-10-28 15:51:41.096319,E,[],009,camera.plate.frames.dev:0:19432,synthetic


In [37]:
plate_detection_silver_df.printSchema()

root
 |-- Time_stamp: timestamp (nullable = true)
 |-- Zone: string (nullable = true)
 |-- detected_plates: string (nullable = true)
 |-- camera_id: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- detector_type: string (nullable = true)



In [38]:
plate_detection_silver_df.write.format("parquet").mode("overwrite").saveAsTable("silver.plate_detections")

print("silver.plate_detections created successfully.")
display(
    spark.sql("SHOW TABLES IN silver").toPandas()
)


silver.plate_detections created successfully.


,namespace,tableName,isTemporary
0,silver,person_detection_details,False
1,silver,person_detections,False
2,silver,plate_detections,False


In [39]:
display(
    employees_df.limit(20).toPandas()
)

,employee_id,plate_number
0,EMP0001,NA
1,EMP0002,SYN-4230
2,EMP0003,SYN-1969
3,EMP0004,SYN-9617
4,EMP0005,NA
5,EMP0006,SYN-9551
6,EMP0007,SYN-2015
7,EMP0008,NA
8,EMP0009,NA
9,EMP0010,SYN-4620


In [40]:
employees_df.printSchema()

root
 |-- employee_id: string (nullable = true)
 |-- plate_number: string (nullable = true)



In [41]:
employees_silver_df = (
    employees_df
    .withColumn(
        "plate_number",
        F.when(
            F.upper(F.trim(F.col("plate_number"))) == "NA",
            F.lit(None).cast("string")
        ).otherwise(F.trim(F.col("plate_number")))
    )
)

In [42]:
display(
    employees_silver_df.limit(20).toPandas()
)

,employee_id,plate_number
0,EMP0001,None
1,EMP0002,SYN-4230
2,EMP0003,SYN-1969
3,EMP0004,SYN-9617
4,EMP0005,None
5,EMP0006,SYN-9551
6,EMP0007,SYN-2015
7,EMP0008,None
8,EMP0009,None
9,EMP0010,SYN-4620


In [43]:
employees_silver_df.write.format("parquet").mode("overwrite").saveAsTable("silver.employees")

In [44]:
(
    cameras_df
    .write
    .format("parquet")
    .mode("overwrite")
    .saveAsTable("silver.cameras")
)

print("silver.cameras created successfully.")

silver.cameras created successfully.


In [46]:
restricted_access_df.printSchema()

root
 |-- employee_id: string (nullable = true)
 |-- zone_id: string (nullable = true)



In [48]:
display(restricted_access_df.toPandas())

,employee_id,zone_id
0,EMP0001,C
1,EMP0011,C
2,EMP0023,C
3,EMP0042,C
4,EMP0048,C
5,EMP0053,C
6,EMP0061,C
7,EMP0067,C
8,EMP0076,C
9,EMP0093,C


In [49]:
restricted_access_df.write.format("parquet").mode("overwrite").saveAsTable("silver.restricted_access")

In [51]:
import sys
from pathlib import Path

ETL_DIR = Path(
    r"D:\projects\AI_Monitoring_Project\advanced\ETL"
)

if str(ETL_DIR) not in sys.path:
    sys.path.insert(0, str(ETL_DIR))

print("ETL directory added to Python path.")

ETL directory added to Python path.


In [52]:
from spark_pipeline.silver.silver_transformation import (
    SilverTransformation
)

silver = SilverTransformation(spark)

silver.run()

STARTING SILVER TRANSFORMATION
Processing person_detections_worker1...
Processing person_detections_worker2...
Successfully written: silver.person_detections
Successfully combined 2 workers.
Successfully written: silver.person_detection_details
Successfully written: silver.plate_detections
Successfully written: silver.employees
Successfully written: silver.cameras
Successfully written: silver.restricted_access
SILVER TRANSFORMATION COMPLETED
